# Benzene Model Hamiltonian: QPE and Resource Estimation

This notebook applies the model Hamiltonian workflow from the Advanced Quantum Chemistry course to **benzene** (C₆H₆).
We derive Extended Hubbard parameters from first-principles molecular integrals, run iterative QPE on a simulator,
and estimate the physical resources required to run the QPE circuit on fault-tolerant hardware.

## Pipeline

| Step | Component | What it does |
|---|---|---|
| Load structure | `Structure` | Molecular geometry input |
| SCF + AVAS | `scf_solver`, `active_space_selector` | Identify π-orbitals |
| Localize | `orbital_localizer` | Define lattice sites |
| Extract parameters | `hamiltonian_constructor` | Read off $t$, $U$ from integrals |
| Build model | `create_ppp_hamiltonian` | Assemble Extended Hubbard $\hat{H}$ |
| Map to qubits | `qubit_mapper` | Jordan-Wigner encoding |
| Run QPE | `phase_estimation` | Iterative QPE on simulator |
| Resource estimation | `circuit.estimate()` | Physical resource costs |

In [ ]:
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

from qdk_chemistry.utils import Logger
Logger.set_global_level(Logger.LogLevel.off)

## 1. Load and Visualize Benzene

Benzene (C₆H₆) has a planar hexagonal ($D_{6h}$) structure with six equivalent C–C bonds.
Its six carbon $p_z$ orbitals form a prototypical aromatic π-system with 6 π-electrons.

In [ ]:
from qdk_chemistry.data import Structure
from qdk_chemistry.algorithms import create
from qdk.widgets import MoleculeViewer

# Load benzene structure from XYZ file
structure = Structure.from_xyz_file(Path("data/benzene.structure.xyz"))

# Visualize the benzene molecule
MoleculeViewer(molecule_data=structure.to_xyz())

## 2. SCF Calculation and π-Orbital Selection

We run Hartree-Fock to obtain molecular orbitals, then use AVAS to automatically select the 6 π-orbitals
by projecting onto the carbon $2p_z$ atomic orbitals. This gives a CAS(6,6) active space:
**6 π-electrons in 6 π-orbitals**.

In [ ]:
# Run SCF
scf_solver = create("scf_solver")
E_hf, wfn_hf = scf_solver.run(structure, charge=0, spin_multiplicity=1, basis_or_guess="sto-3g")
print(f"Hartree-Fock energy: {E_hf:.6f} Hartree")

In [ ]:
import qdk_chemistry.plugins.pyscf  # Enable PySCF plugin  # noqa: F401

# Use AVAS to select the π-orbitals by projecting onto the carbon 2pz atomic orbitals
avas_selector = create("active_space_selector", "pyscf_avas", ao_labels=["C 2pz"])
avas_wfn = avas_selector.run(wfn_hf)
indices, _ = avas_wfn.get_orbitals().get_active_space_indices()
n_active_e = sum(avas_wfn.get_active_num_electrons())
print(f"AVAS selected CAS({n_active_e},{len(indices)}) active space")
print(f"  Active orbital indices: {list(indices)}")

### Visualize the AVAS-selected π-orbitals

For benzene's 6-site cyclic π-system, we expect the familiar Hückel pattern:
one orbital with no nodes, two degenerate orbitals with one node each, two degenerate orbitals
with two nodes, and one orbital with three nodes.

In [ ]:
from qdk_chemistry.utils.cubegen import generate_cubefiles_from_orbitals

# Visualize the AVAS-selected active space orbitals
active_orbitals = avas_wfn.get_orbitals()
cube_data = generate_cubefiles_from_orbitals(
    orbitals=active_orbitals,
    grid_size=(40, 40, 40),
    margin=10.0,
    indices=indices,
)

MoleculeViewer(molecule_data=structure.to_xyz(), cube_data=cube_data)

## 3. Localize Orbitals to Define Lattice Sites

We apply Pipek-Mezey localization to transform the 6 delocalized π-orbitals into
6 site-localized orbitals — one centered on each carbon atom.
These localized orbitals serve as the "sites" in our Extended Hubbard model.

In [ ]:
# Localize the active-space orbitals
localizer = create("orbital_localizer", "qdk_pipek_mezey")
loc_wfn = localizer.run(avas_wfn, *avas_wfn.get_orbitals().get_active_space_indices())

# Visualize the localized orbitals
loc_orbitals = loc_wfn.get_orbitals()
loc_indices, _ = loc_orbitals.get_active_space_indices()
cube_data = generate_cubefiles_from_orbitals(
    orbitals=loc_orbitals,
    grid_size=(40, 40, 40),
    margin=10.0,
    indices=loc_indices,
)

MoleculeViewer(molecule_data=structure.to_xyz(), cube_data=cube_data)

## 4. Extract Extended Hubbard Parameters

From the localized orbitals we read off the model parameters directly from the *ab initio* integrals:
- **$t$ (hopping)**: largest off-diagonal one-body integral for each orbital
- **$U$ (on-site repulsion)**: diagonal two-body integrals $\langle ii|ii \rangle$
- **$\varepsilon$ (on-site energy)**: set to zero

In [ ]:
n_sites = len(indices)
n_active_alpha, n_active_beta = avas_wfn.get_active_num_electrons()

# Construct the active-space Hamiltonian from the localized orbitals
hamiltonian_constructor = create("hamiltonian_constructor")
refined_orbitals = loc_wfn.get_orbitals()
active_hamiltonian = hamiltonian_constructor.run(refined_orbitals)

# Get one-body integrals of the active space
h1_alpha, _ = active_hamiltonian.get_one_body_integrals()
print(f"One-body integrals ({n_sites}x{n_sites}):")
print(np.round(h1_alpha, 4))

# Hopping integral: identify nearest-neighbour hoppings
nn_hoppings = []
for i in range(n_sites):
    row = np.abs(h1_alpha[i].copy())
    row[i] = 0.0  # exclude diagonal
    nn_hoppings.append(np.max(row))
t = float(np.mean(nn_hoppings))

# On-site Coulomb repulsion U: average of diagonal two-body integrals (i,i,i,i)
U_values = []
for i in range(n_sites):
    U_values.append(active_hamiltonian.get_two_body_element(i, i, i, i))
U = float(np.mean(U_values))

print(f"\nExtended Hubbard parameters from CAS({n_active_alpha + n_active_beta},{n_sites}) active space:")
print(f"  Hopping integral t = {t:.4f} Hartree")
print(f"  On-site repulsion U = {U:.4f} Hartree")
print(f"  Ratio U/t = {U / t:.2f}")

## 5. Build the Lattice Hamiltonian

Benzene's π-system has a **ring topology**: each carbon is bonded to two neighbors, forming a
cyclic 6-site chain. We build a periodic lattice and compute the nearest-neighbor repulsion $V$
using the Ohno potential.

In [ ]:
from qdk_chemistry.data import LatticeGraph
from qdk_chemistry.utils.model_hamiltonians import create_ppp_hamiltonian, ohno_potential

# Build periodic chain lattice (ring topology for benzene)
lattice = LatticeGraph.chain(n=n_sites, periodic=True)

# Compute nearest-neighbour C-C bond length from the structure
coords = structure.get_coordinates()
carbon_coords = coords[:n_sites]  # first n_sites atoms are carbons
cc_distances = [np.linalg.norm(carbon_coords[i] - carbon_coords[(i + 1) % n_sites]) for i in range(n_sites)]
R_CC = float(np.mean(cc_distances))
print(f"Average C-C bond length: {R_CC:.4f} Bohr")

# Compute nearest-neighbour intersite Coulomb repulsion V via Ohno potential
V = ohno_potential(lattice, U=U, R=R_CC, nearest_neighbor_only=True)

# Build Extended Hubbard Hamiltonian
hamiltonian = create_ppp_hamiltonian(lattice, epsilon=0.0, t=t, U=U, V=V, z=1.0)

# Display model summary
h1 = hamiltonian.get_one_body_integrals()[0]
Vij = V[0, 1]

print(f"\nExtended Hubbard model: {lattice.num_sites} sites (ring), {n_active_alpha}α + {n_active_beta}β electrons")
print(f"One-body integrals:\n{np.round(h1, 4)}")
print(f"Two-body integrals (U): {U:.4f}")
print(f"Two-body integrals (V): {Vij:.4f}")
print(f"Ratio U/t = {U / t:.2f}, V/t = {Vij / t:.2f}")

## 6. Classical Reference: Exact Diagonalization

With 6 sites, the model is small enough to solve exactly via CASCI.
This gives a classical reference energy and reveals the multi-reference character of the ground state.

In [ ]:
from qdk.widgets import Histogram

# Exact diagonalization (CASCI)
mc_calculator = create("multi_configuration_calculator", "macis_cas")
e_exact, wfn_exact = mc_calculator.run(hamiltonian, n_active_alpha, n_active_beta)
print(f"Exact ground state energy: {e_exact:.6f} Hartree")

# Plot top configuration weights
num_configurations = len(wfn_exact.get_active_determinants())
print(f"Total configurations: {num_configurations}")
top_dets = wfn_exact.get_top_determinants()
display(
    Histogram(
        bar_values={k.to_string(): np.abs(v)**2 for k, v in top_dets.items()},
        items="top-25",
        sort="high-to-low",
    )
)

## 7. Trial State Preparation

Benzene's ground state is spread across many configurations due to its high symmetry.
We use the top 50 determinants from the exact wavefunction to build a trial state
and prepare it as a quantum circuit using the GF2+X sparse isometry method.

In [ ]:
from qdk_chemistry.data import Wavefunction, CasWavefunctionContainer

NUM_TRIAL_DETS = 50

# Select top determinants and renormalize
dets = wfn_exact.get_top_determinants(NUM_TRIAL_DETS)
total_weight = sum(np.abs(c)**2 for c in dets.values())
dets = {det: c / np.sqrt(total_weight) for det, c in dets.items()}

# Build trial wavefunction
det_keys = list(dets.keys())
coeffs = np.array(list(dets.values()))
wfn_trial = Wavefunction(CasWavefunctionContainer(coeffs, det_keys, wfn_exact.get_orbitals()))

# Compute fidelity with exact wavefunction
exact_coeffs = np.array([wfn_exact.get_coefficient(det) for det in det_keys])
fidelity = np.abs(np.vdot(exact_coeffs, coeffs))**2
print(f"Overlap (fidelity) of trial state with CASCI wavefunction: {fidelity:.4f}")

# Plot trial state configuration weights
display(
    Histogram(
        bar_values={k.to_string(): np.abs(v)**2 for k, v in dets.items()},
        sort="high-to-low",
    )
)

In [ ]:
from qdk.widgets import Circuit

# Generate state preparation circuit
state_prep = create("state_prep", "sparse_isometry_gf2x")
state_prep_circuit = state_prep.run(wfn_trial)

# Visualize the circuit
display(Circuit(state_prep_circuit.get_qsharp_circuit()))

## 8. Quantum Phase Estimation

We run iterative QPE on the 12-qubit model Hamiltonian (6 sites × 2 spins).
Compare this to the full *ab initio* Hamiltonian which would require significantly more qubits.

In [ ]:
# Map to qubit Hamiltonian
qubit_mapper = create("qubit_mapper", algorithm_name="qdk", encoding="jordan-wigner")
qubit_hamiltonian = qubit_mapper.run(hamiltonian)
print("Qubit Hamiltonian:\n", qubit_hamiltonian.get_summary())

In [ ]:
# Set up iQPE parameters
M_PRECISION = 8
SHOTS_PER_BIT = 3
SIMULATOR_SEED = 42

evolution_time = np.pi / qubit_hamiltonian.schatten_norm
print(f"Number of qubits: {qubit_hamiltonian.num_qubits}")
print(f"Number of Pauli terms: {len(qubit_hamiltonian.pauli_strings)}")
print(f"Evolution time: {evolution_time:.4f} Hartree⁻¹")

In [ ]:
# Build iQPE components
evolution_builder = create("time_evolution_builder", "trotter", order=2)
circuit_mapper = create("controlled_evolution_circuit_mapper", "pauli_sequence")
iqpe = create(
    "phase_estimation",
    "iterative",
    num_bits=M_PRECISION,
    evolution_time=evolution_time,
    shots_per_bit=SHOTS_PER_BIT,
)

# Visualize one iQPE iteration circuit
iqpe_iter_circuit = iqpe.create_iteration_circuit(
    state_preparation=state_prep_circuit,
    qubit_hamiltonian=qubit_hamiltonian,
    evolution_builder=evolution_builder,
    circuit_mapper=circuit_mapper,
    iteration=M_PRECISION - 3,
    total_iterations=M_PRECISION,
)

display(Circuit(iqpe_iter_circuit.get_qsharp_circuit()))

### Run iQPE

In [ ]:
# Execute iQPE
circuit_executor = create("circuit_executor", "qdk_full_state_simulator", seed=SIMULATOR_SEED)
result = iqpe.run(
    state_preparation=state_prep_circuit,
    qubit_hamiltonian=qubit_hamiltonian,
    circuit_executor=circuit_executor,
    evolution_builder=evolution_builder,
    circuit_mapper=circuit_mapper,
)

# Compare with exact result
estimated_energy = result.raw_energy + hamiltonian.get_core_energy()
estimated_error = abs(estimated_energy - e_exact)
print(f"QPE Results ({M_PRECISION}-bit precision):")
print(f"  Reference CASCI energy:    {e_exact:.6f} Hartree")
print(f"  QPE estimated energy:      {estimated_energy:.6f} Hartree")
print(f"  Energy difference:         {estimated_error:.3e} Hartree")

## 9. Resource Estimation

How many physical qubits and how much runtime would be needed to run this QPE circuit on
fault-tolerant hardware? We use the QDK Resource Estimator to answer this question.

We generate the most expensive iQPE iteration circuit (iteration 0, which applies $U^{2^{M-1}}$)
and estimate resources for the surface code with gate-based superconducting qubits.

In [ ]:
from qdk.estimator import EstimatorParams, QubitParams, QECScheme

# Generate the most expensive iQPE iteration circuit (iteration 0)
iqpe_circuit_for_re = iqpe.create_iteration_circuit(
    state_preparation=state_prep_circuit,
    qubit_hamiltonian=qubit_hamiltonian,
    evolution_builder=evolution_builder,
    circuit_mapper=circuit_mapper,
    iteration=0,
    total_iterations=M_PRECISION,
)

# Configure resource estimation parameters
params = EstimatorParams()
params.error_budget = 0.01
params.qubit_params.name = QubitParams.GATE_NS_E3
params.qec_scheme.name = QECScheme.SURFACE_CODE

# Run resource estimation
re_result = iqpe_circuit_for_re.estimate(params)

# Display summary
physical = re_result["physicalCounts"]
breakdown = physical["breakdown"]
print("=== Resource Estimation Summary ===")
print(f"  Physical qubits:      {physical['physicalQubits']:,}")
print(f"  Algorithmic qubits:   {breakdown['algorithmicLogicalQubits']}")
print(f"  Runtime:              {physical['runtime']}")
print(f"  T gates:              {breakdown.get('numTstates', 'N/A')}")

### Resource Estimation Dashboard

The interactive widget below provides a detailed breakdown of qubit utilization,
the space-time trade-off, and the full resource report.

In [ ]:
from qdk.widgets import EstimateDetails

# Display detailed resource estimation report
display(EstimateDetails(re_result))

## Summary

In this notebook we:

1. **Derived Extended Hubbard parameters for benzene** from first-principles molecular integrals using AVAS + Pipek-Mezey localization
2. **Built a 6-site ring model** with the Ohno potential for nearest-neighbor repulsion
3. **Ran iterative QPE** on the model Hamiltonian using only 12 qubits (6 sites × 2 spins)
4. **Estimated physical resources** for running QPE on fault-tolerant hardware with surface codes

The model Hamiltonian approach dramatically reduces qubit requirements while retaining the essential π-electron physics of benzene.